# Evaluation, health: rerun the utility questions on a finished run (Kaggle, ~1 hour)

The first utility questions asked about a record's *wording* (ChatDoctor questions quoted the
patient; scan questions named hospitals and section headings), which the rewrite changes, so
ours often answered "The records do not say" with the right record in hand. This notebook asks
new questions about the *content* (`tools/eval_suite/utility_questions.py`) on the stores your
5h run already built, and writes a new report. Nothing is rebuilt; every other number is the
same as in the 5h run.

**Before running:**
1. Session options: **GPU T4 x2**, Internet **On**; secret **GEMINI_API_KEY** attached.
2. Right sidebar → **Input** → **+ Add Input** → **Your Work** → pick the evaluation notebook
   whose 5h version finished → **Add**. Its output (the stores, answers and questions) appears
   under `/kaggle/input/`.
3. **Save Version → Save & Run All (Commit)**, then close Kaggle. When it finishes, the
   version's **Output** tab holds `eval_results_health_5h.zip` (the new full report).

| Cell | What it does | Time |
|---|---|---|
| 1 Setup | clone main, install, Ollama + llama3:8b (no vision model needed) | ~12 min |
| 2 Restore | copy the 5h run from the attached input | ~1 min |
| 3 New questions | Gemini rewrites 100 ChatDoctor and 50 scan questions about content | ~12 min |
| 4 Answer | the new utility questions, both pipelines, on the saved stores | ~15 min |
| 5 End to end | the scans' new questions on both scan stores | ~10 min |
| 6 Report + pack | the full report (all metrics) and the zip | ~3 min |

In [ ]:
# 1 Setup (run at the start of every session)
SCALE = "5h"  # the run being redone; keep "5h"   # "5h": every metric on less data, ~4.5 h; "quick": ~10 min check; "full": the plan's volumes (~12-15 h)
MODEL = "llama3:8b"                       # answers, both pipelines (as the dashboard)
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.1-flash-lite"   # Agents 1-3 (fallback order)
OCR_MODEL = "qwen3-vl:8b-instruct"        # reads the scans (end-to-end check)
BRANCH = "main"

import os, subprocess, sys
from pathlib import Path

W = Path("/kaggle/working")
REPO = W / "3rd_sem_el"
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"      # models stay out of /kaggle/working
os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # answers are asked 2 questions x 2 pipelines at once
os.environ["EVAL_CONCURRENCY"] = "2"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5" matplotlib
sys.path.insert(0, str(REPO / "tools"))
import colab_cache
colab_cache.install_ollama_and_model(MODEL)
colab_cache.start_ollama()

from kaggle_secrets import UserSecretsClient
try:
    key = UserSecretsClient().get_secret("GEMINI_API_KEY")
except Exception as exc:
    raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets), tick it as "
                       "attached to this notebook, and run this cell again.") from exc
WORK = W / "eval" / ("health" if SCALE == "full" else f"health_{SCALE}")
os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama", "SANITISE_MODEL": MODEL, "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid", "SANITISE_OCR_MODEL": OCR_MODEL,
    "SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
    "SANITISE_GEMINI_API_KEY": key, "GEMINI_API_KEYS": key,
    "SANITISE_MONITOR_DB_PATH": str(WORK / "stores" / "monitor.sqlite"),
})
RUN = f"python tools/eval_suite/run.py {{step}} --domain health --work {WORK} --scale {SCALE}"
def run(step, extra=""):
    colab_cache.start_ollama()  # back up if it stopped (e.g. a cell was interrupted)
    cmd = RUN.format(step=step) + " " + extra
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)
print(f"ready: answers {MODEL} | Agents 1-3 {GEMINI_MODEL} ({len(key.split(','))} key(s)) | "
      f"OCR {OCR_MODEL} | results in {WORK}")

In [ ]:
# 2 Restore the finished 5h run from the attached input (its stores, answers and questions)
import shutil
found = [p.parent for p in Path("/kaggle/input").rglob("stores") if p.parent.name == "health_5h"]
if not found:
    raise RuntimeError("No finished 5h run under /kaggle/input: add the evaluation notebook's "
                       "output as an input (right sidebar -> Input -> + Add Input -> Your Work).")
if not (WORK / "stores" / "real.sqlite").exists():
    shutil.copytree(found[0], WORK, dirs_exist_ok=True)
print("restored", found[0], "->", WORK, "| stores:", sorted(p.name for p in (WORK / "stores").iterdir()))

In [ ]:
# 3 New utility questions about content (Gemini): ChatDoctor and the scans
run("utility-questions")

In [ ]:
# 4 Answer the new ChatDoctor utility questions, both pipelines, on the saved real store
run("single", "--only utility")

In [ ]:
# 5 Answer the scans' new utility questions on both scan stores (gold text and OCR)
run("e2e")

In [ ]:
# 9 Report: every metric, REPORT.md and the figures
run("report")
from IPython.display import Image, Markdown, display
display(Markdown((WORK / "results" / "REPORT.md").read_text()))
for fig in sorted((WORK / "results" / "figures").glob("*.png")):
    display(Image(str(fig)))

In [ ]:
# 10 Pack the results to download (Output tab, or Save Version) and commit to the repo
import shutil, zipfile
zip_path = W / f"eval_results_health{'' if SCALE == 'full' else '_' + SCALE}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/utility.json", "e2e/labels.json", "e2e/manifest.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"health/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output tab (or Save Version), then follow 'Saving the results' in the guide.")